# Groq policy responses for PASS, REJECTED, and REVIEW cases

This notebook calls Groq once for each synthetic case. It reads `loan_policy.md` at runtime. Knowlytix `ClaimVerifier` computes the policy screen before Groq is asked to explain it.

**Live API calls may use Groq quota.** Responses are informational. A PASS means only eligible for underwriting; the policy's Approval Authority and a Loan Officer retain decision authority. When tension calibration is degenerate, valid structured responses can be shown as reviewer-only previews; they are not released to borrowers or used to make decisions.


In [50]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import loanguard as loan

load_dotenv(PROJECT / ".env", override=False)
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in the project .env or process environment.")
print("Groq key detected; value is hidden.")


Groq key detected; value is hidden.


## 1. Check the live model catalog


In [51]:
catalog = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=30,
)
catalog.raise_for_status()
groq_model_ids = sorted(item["id"] for item in catalog.json()["data"])
print("Models available to this key:")
for model_id in groq_model_ids:
    print(" -", model_id)


Models available to this key:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo


In [52]:
from knowlytix.knowledge.llm_backend import LiteLLMBackend

requested_model = (os.getenv("KNOWLYTIX_GROQ_MODEL") or "").strip().removeprefix("groq/")
if requested_model:
    raw_model_id = requested_model
else:
    preferred_models = [
        "openai/gpt-oss-20b", "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant", "qwen/qwen3-32b",
    ]
    raw_model_id = next((model for model in preferred_models if model in groq_model_ids), None)
    if raw_model_id is None:
        raise RuntimeError(
            "No preferred chat model is available. Set KNOWLYTIX_GROQ_MODEL "
            "to an exact model ID from the catalog and rerun this cell."
        )
    print("No model override set; selected available model:", raw_model_id)
if raw_model_id not in groq_model_ids:
    raise ValueError(f"{raw_model_id!r} is not available in the live catalog.")
groq_llm = LiteLLMBackend(f"groq/{raw_model_id}")
print("Configured Knowlytix backend:", groq_llm.model_name)


No model override set; selected available model: openai/gpt-oss-20b
Configured Knowlytix backend: groq/openai/gpt-oss-20b


## 2. Load policy knowledge and calibrate the verifier

The same calibration from `build_verifier()` is used for typed policy claims and to configure the optional geometric gate. If tension calibration is unusable, the geometric gate is skipped explicitly; a theoretical or hardcoded threshold is never substituted.


In [53]:
import importlib
loan = importlib.reload(loan)  # Load the current project helper.

from knowlytix.harness._verifier import LogicVerifier
from knowlytix.harness.governance.governed_llm import ConstrainedLLM
from knowlytix.knowledge.config import VerifyConfig

store = loan.build_or_load_store(rebuild=False, epochs=100)
claim_verifier = loan.build_verifier(store)
tension_calibration = claim_verifier.calibrated_thresholds
active_plausibility_cut = claim_verifier.plausibility_threshold
active_plausibility_ci = claim_verifier.plausibility_interval
print({
    'tension_status': tension_calibration.tension_status,
    'tau_ent': tension_calibration.tau_ent,
    'tau_contra': tension_calibration.tau_contra,
    'active_claim_verifier_plausibility_cut': active_plausibility_cut,
    'active_claim_verifier_bootstrap_ci': active_plausibility_ci,
    'judge_single_fit_plausibility_diagnostic_only': tension_calibration.tau_plausibility,
})

constrained = None
if tension_calibration.tension_ok and active_plausibility_cut is not None:
    logic_verifier = LogicVerifier(
        store, VerifyConfig(
            tau_ent=tension_calibration.tau_ent,
            tau_contra=tension_calibration.tau_contra,
            min_plausibility=active_plausibility_cut,
        ),
    )
    constrained = ConstrainedLLM(
        store=store, verifier=logic_verifier, llm=groq_llm,
        allowed_tools=['lookup_product_terms'],
    )
    print('Geometric claim gate: configured; all responses remain reviewer-only previews.')
else:
    print('Geometric claim gate unavailable: tension calibration is degenerate; narrative extraction is skipped.')


  GMS entities:  52
  GMS relations: 11
  GMS triples:   53
  Store loaded from N:\Solytics\Loan_Approval\loan_store
  Entities:  52
  Relations: 11
  Triples:   53
  ENM:       18
  Documents: 1
[1] loaded existing store from N:\Solytics\Loan_Approval\loan_store
  Calibrated geodesic      :   106 samples, 2-class, train accuracy=59.4%  τ=1.5194694393873216
  Calibrated tension       :   530 samples, 2-class, train accuracy=94.3%  τ=DEGENERATE (no usable two-cut separation)
[2] tau_plausibility=1.4921 (95% CI [1.2627, 1.6926]), tension_status=degenerate
{'tension_status': 'degenerate', 'tau_ent': None, 'tau_contra': None, 'active_claim_verifier_plausibility_cut': 1.4921228142219538, 'active_claim_verifier_bootstrap_ci': (1.2626728211827576, 1.692587912928313), 'judge_single_fit_plausibility_diagnostic_only': 1.5194694393873216}
Geometric claim gate unavailable: tension calibration is degenerate; narrative extraction is skipped.


## 3. Run the three examples through the policy checks

The sample values are scenario inputs. The policy limits are read from the GMS ENM entries created from `loan_policy.md`. A failed policy check is labeled REJECTED by the pre-screen. This is not a final loan decision; the Loan Officer named in the policy retains decision authority.


In [54]:
examples = [
    {"example": "PASS", "applicant_id": "DEMO-PASS", "product": "Personal Loan",
     "requested_amount": 12000, "credit_score": 712, "proposed_apr": 16.99},
    {"example": "REJECTED", "applicant_id": "DEMO-REJECTED", "product": "Personal Loan",
     "requested_amount": 28000, "credit_score": 735, "proposed_apr": 18.50},
    {"example": "REVIEW", "applicant_id": "DEMO-REVIEW", "product": "Personal Loan",
     "requested_amount": 15000, "credit_score": 700, "proposed_apr": None},
]
example_results = []
for example in examples:
    screened = loan.screen_application(store, claim_verifier, example)
    example_results.append({**example, **screened})

status_rows = []
for result in example_results:
    checks = result["checks"]
    status_rows.append({
        "example": result["example"],
        "screening_status": result["screening_status"],
        "screening_result": result["screening_result"],
        "amount_check": checks["requested_amount_within_policy_maximum"]["status"],
        "score_check": checks["credit_score_meets_policy_minimum"]["status"],
        "APR_check": checks["proposed_apr_within_policy_maximum"]["status"],
        "income_verification_required": result["income_verification_required"],
        "routing": result["routing"],
        "human_decision_required": result["human_decision_required"],
    })
pd.DataFrame(status_rows)


,example,screening_status,screening_result,amount_check,score_check,APR_check,income_verification_required,routing,human_decision_required
0,PASS,PASS,PASSED_TO_UNDERWRITING,PASS,PASS,PASS,True,POLICY_ELIGIBLE_FOR_UNDERWRITING,True
1,REJECTED,REJECTED,REJECTED_BY_POLICY_SCREEN,FAIL,PASS,PASS,True,PRE_SCREEN_REJECTED,True
2,REVIEW,REVIEW,REVIEW_REQUIRED,PASS,PASS,REVIEW,True,MANUAL_REVIEW,True


## 4. Ask Groq to explain each result using the Markdown policy

Each request contains the full policy text and the actual Knowlytix check evidence for that example. The prompt asks for one short paragraph to reduce truncation. `LiteLLMBackend.call()` returns text but not Groq `finish_reason`, so the punctuation indicator below is only a visual warning, not authoritative completion metadata.


In [55]:
policy_text = (PROJECT / "loan_policy.md").read_text(encoding="utf-8-sig")
drafts = []
for result in example_results:
    checks = result["checks"]
    expected = {
        "example": result["example"],
        "screening_status": result["screening_status"],
        "screening_result": result["screening_result"],
        "amount_check": checks["requested_amount_within_policy_maximum"]["status"],
        "score_check": checks["credit_score_meets_policy_minimum"]["status"],
        "apr_check": checks["proposed_apr_within_policy_maximum"]["status"],
        "income_verification_required": result["income_verification_required"],
        "routing": result["routing"],
        "human_decision_required": result["human_decision_required"],
    }
    scenario = {
        "application": {
            "product": result["product"],
            "requested_amount": result["requested_amount"],
            "credit_score": result["credit_score"],
            "proposed_apr": result["proposed_apr"],
        },
        "knowlytix_verified_result": expected,
        "policy_source": policy_text,
    }
    try:
        response = groq_llm.call(
            system=(
                "Return one valid JSON object only, without Markdown fences. Copy every "
                "screening and check field exactly from knowlytix_verified_result; do "
                "not reinterpret, combine, or change any status. Add one short explanation "
                "sentence based only on the supplied policy and verified result. "
                "Make it case-specific by naming checks that passed, failed, or need review. "
                "A PASS means eligible for underwriting, never final approval; REJECTED means a policy pre-screen failed, not a final decline; "
                "not a final decline. Do not call any role the sole authority or invent "
                "approval powers beyond the policy's Approval Authority field."
            ),
            user=(
                "Explain this case. Return exactly these fields: example, screening_status, "
                "screening_result, amount_check, score_check, apr_check, "
                "income_verification_required, routing, human_decision_required, explanation. "
                "Use the verified values exactly. Make the explanation vary with this "
                "applicant's actual check results; avoid a generic repeated sentence. "
                "Do not add facts absent from the policy or verified result.\n\n"
                + json.dumps(scenario, indent=2)
            ),
            max_tokens=1800,
        )
        parsed = json.loads(response.strip())
        mismatches = [key for key, value in expected.items() if parsed.get(key) != value]
        valid_json = True
    except Exception as exc:
        parsed = None
        mismatches = ["invalid_or_incomplete_json"]
        valid_json = False
        error = type(exc).__name__
    else:
        error = None
    drafts.append({
        "example": result["example"],
        "expected": expected,
        "parsed": parsed,
        "response": response if error is None else "",
        "valid_json": valid_json,
        "mismatched_fields": mismatches,
        "provider_error": error,
    })
    print(f"\n### {result['example']} — policy status {result['screening_status']}")
    if error:
        print("Groq response could not be parsed:", error)
    else:
        print(json.dumps(parsed, indent=2))
        print("Status field mismatches:", mismatches or "none")

pd.DataFrame([
    {"example": row["example"], "valid_json": row["valid_json"],
     "mismatched_fields": row["mismatched_fields"], "provider_error": row["provider_error"]}
    for row in drafts
])



### PASS — policy status PASS
{
  "example": "PASS",
  "screening_status": "PASS",
  "screening_result": "PASSED_TO_UNDERWRITING",
  "amount_check": "PASS",
  "score_check": "PASS",
  "apr_check": "PASS",
  "income_verification_required": true,
  "routing": "POLICY_ELIGIBLE_FOR_UNDERWRITING",
  "human_decision_required": true,
  "explanation": "The applicant meets all screening criteria\u2014amount, credit score, and APR are within limits\u2014and income verification is required; the case is routed to underwriting with a human decision pending."
}
Status field mismatches: none

### REJECTED — policy status REJECTED
{
  "example": "REJECTED",
  "screening_status": "REJECTED",
  "screening_result": "REJECTED_BY_POLICY_SCREEN",
  "amount_check": "FAIL",
  "score_check": "PASS",
  "apr_check": "PASS",
  "income_verification_required": true,
  "routing": "PRE_SCREEN_REJECTED",
  "human_decision_required": true,
  "explanation": "The pre\u2011screen rejected because the requested amount of 

,example,valid_json,mismatched_fields,provider_error
0,PASS,True,[],None
1,REJECTED,True,[],None
2,REVIEW,True,[],None


## 5. Validate response fields and show reviewer-only previews

The JSON response is checked against the deterministic policy screen. The claim extractor must receive only the natural-language `explanation`, never the JSON field names. With the current degenerate tension calibration, extraction is skipped and field-valid output is labeled a reviewer-only preview. This notebook has no borrower-facing release or automatic loan approval path.


In [56]:
import json
results_by_example = {result['example']: result for result in example_results}
gate_rows = []
for item in drafts:
    result = results_by_example[item['example']]
    if item['provider_error'] or not item['response'].strip():
        gate_rows.append({'example': item['example'], 'screening_status': result['screening_status'],
                          'geometric_claim_gate': 'NOT RUN: no valid response',
                          'claims_checked': 'NOT RUN', 'llm_response_release_status': 'BLOCK_INVALID_RESPONSE'})
        continue
    try:
        parsed = json.loads(item['response'])
    except (TypeError, json.JSONDecodeError):
        gate_rows.append({'example': item['example'], 'screening_status': result['screening_status'],
                          'geometric_claim_gate': 'NOT RUN', 'claims_checked': 'NOT RUN',
                          'llm_response_release_status': 'BLOCK_INVALID_RESPONSE'})
        continue
    expected = {
        'example': result['example'], 'screening_status': result['screening_status'],
        'screening_result': result['screening_result'],
        'amount_check': result['checks']['requested_amount_within_policy_maximum']['status'],
        'score_check': result['checks']['credit_score_meets_policy_minimum']['status'],
        'apr_check': result['checks']['proposed_apr_within_policy_maximum']['status'],
        'income_verification_required': result['income_verification_required'],
        'routing': result['routing'], 'human_decision_required': result['human_decision_required'],
    }
    fields_match = all(parsed.get(k) == v for k, v in expected.items())
    explanation = parsed.get('explanation')
    fields_match = fields_match and isinstance(explanation, str) and bool(explanation.strip())
    gate_available = bool(constrained is not None and tension_calibration.tension_ok)
    # Never treat a reviewer preview as a release, even if future extraction is enabled.
    if fields_match and gate_available:
        extracted_claims = constrained.extract_claims(explanation)
        claims_checked = f'{len(extracted_claims)} claims extracted from explanation only'
        gate_status = 'AVAILABLE; reviewer preview only'
    elif fields_match:
        claims_checked = 'NOT RUN (tension calibration unavailable)'
        gate_status = 'UNAVAILABLE: tension calibration degenerate'
    else:
        claims_checked = 'NOT RUN (structured fields failed validation)'
        gate_status = 'NOT RUN: invalid structured response'
    gate_rows.append({
        'example': item['example'], 'screening_status': result['screening_status'],
        'applicant_policy_outcome': result['routing'],
        'fields_match_policy_screen': fields_match, 'geometric_claim_gate': gate_status,
        'claims_checked': claims_checked,
        'llm_response_release_status': 'PREVIEW_FOR_HUMAN_REVIEW' if fields_match else 'BLOCK_INVALID_RESPONSE',
    })
pd.DataFrame(gate_rows)


,example,screening_status,applicant_policy_outcome,fields_match_policy_screen,geometric_claim_gate,claims_checked,llm_response_release_status
0,PASS,PASS,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,UNAVAILABLE: tension calibration degenerate,NOT RUN (tension calibration unavailable),PREVIEW_FOR_HUMAN_REVIEW
1,REJECTED,REJECTED,PRE_SCREEN_REJECTED,True,UNAVAILABLE: tension calibration degenerate,NOT RUN (tension calibration unavailable),PREVIEW_FOR_HUMAN_REVIEW
2,REVIEW,REVIEW,MANUAL_REVIEW,True,UNAVAILABLE: tension calibration degenerate,NOT RUN (tension calibration unavailable),PREVIEW_FOR_HUMAN_REVIEW


## Interpretation

The typed threshold results are policy checks from Knowlytix; Groq only explains those results. PASS means eligible for the human underwriting stage, REJECTED means a policy threshold failed at pre-screen, and REVIEW means a required value could not be checked. None is a final credit decision. With the current degenerate calibration, the geometric LLM claim gate stays fail-closed unless the store produces usable labeled-pair tension calibration.
